<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Random SFI API Test

Each test chooses a random part **and** a random action, and each action gets its own backend-style reply. There are 3 actions and 3 parts, so 9 different pairs are possible.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Random SFI API Test

import random

actions = [
    "POST/create",
    "PUT/update",
    "GET/search"
]

parts = [
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"},
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"}
]

for test in range(1, 6):
    part = random.choice(parts)
    action = random.choice(actions)

    if action == "POST/create":
        response = "201 Created - " + part["product_name"] + " passed checks and was saved"
    elif action == "PUT/update":
        response = "200 OK - changes saved to spec " + part["spec_number"]
    else:  # GET/search
        response = "200 OK - located spec " + part["spec_number"]

    print("Test", test, "|", part["product_name"], "|", action, "->", response)

## Popcorn Hack 2: Random Structured SFI Car Part

Because `random.choice(parts)` grabs a full record instead of single values, the category, spec number, and product name of the part it picks never get mixed up.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random SFI Part Record

import random

parts = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    }
]

picked_part = random.choice(parts)

print("SFI part picked for testing:")
print("  Category:    ", picked_part["category"])
print("  Spec number: ", picked_part["spec_number"])
print("  Product name:", picked_part["product_name"])

## Popcorn Hack 3: Random SFI Record Number

There are 4 records, numbered 1 up to 4. Since `random.randint(1, record_count)` includes both 1 and 4, no record is left out. To check, the runner makes 100 picks and counts how many times each record shows up. Repeats happen because every pick starts fresh and ignores the one before.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random SFI Record Number

import random

record_count = 4

record_id = random.randint(1, record_count)
print("SFI record chosen for QA:", record_id)

# Check that every valid record ID is possible: make 100 picks and count them.
counts = {record: 0 for record in range(1, record_count + 1)}
for pick in range(100):
    counts[random.randint(1, record_count)] += 1

print()
print("How often each record was picked (100 runs):")
for record, count in counts.items():
    print("  Record", record, "->", count)
print("No record was skipped:", all(count > 0 for count in counts.values()))
print("Possible outputs:", list(range(1, record_count + 1)))

## Popcorn Hack 4 / Homework: SFI Backend QA Simulator

A simulator you can reuse that runs `test_count` random tests. Every test picks a random part and a random action, then applies the action to the stored spec numbers:

| Action | Spec not stored | Spec already stored |
| --- | --- | --- |
| `POST/create` | 201: created and stored | 409: duplicate spec, rejected |
| `DELETE/remove` | 404: nothing to remove | 200: removed |
| `GET/search` | 404: not found | 200: record found |
| `PUT/update` | 404: nothing to update | 200: updated |

The stored list is updated as the tests go. So if a spec is removed early on, a later search for it comes back 404, and a spec created early is refused if it is created a second time.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend QA Simulator

import random

actions = [
    "POST/create",
    "DELETE/remove",
    "GET/search",
    "PUT/update"
]

parts = [
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    }
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 7


def run_qa_test(part, action):
    """Send one action to the backend and return the response."""
    spec = part["spec_number"]
    stored = spec in existing_spec_numbers

    if action == "POST/create":
        if not stored:
            existing_spec_numbers.append(spec)
            return "201 Created: added spec " + spec
        return "409 Conflict: spec " + spec + " is a duplicate"

    if action == "DELETE/remove":
        if not stored:
            return "404 Not Found: spec " + spec + " cannot be removed"
        existing_spec_numbers.remove(spec)
        return "200 OK: deleted spec " + spec

    if action == "GET/search":
        if not stored:
            return "404 Not Found: spec " + spec + " does not exist"
        return "200 OK: located " + part["product_name"]

    if action == "PUT/update":
        if not stored:
            return "404 Not Found: spec " + spec + " cannot be updated"
        return "200 OK: saved changes to spec " + spec

    return "400 Bad Request: " + action + " is not a known action"


def run_simulator(count):
    print("Stored specs before testing:", existing_spec_numbers)
    print()
    for test_number in range(1, count + 1):
        part = random.choice(parts)
        action = random.choice(actions)
        result = run_qa_test(part, action)
        print("Test", test_number, "|", part["spec_number"], part["product_name"], "|", action)
        print("   ->", result)
    print()
    print("Stored specs after testing:", existing_spec_numbers)
    print("Every stored spec is unique:", len(existing_spec_numbers) == len(set(existing_spec_numbers)))


run_simulator(test_count)